# Case study: Tides, storm surge and sea level

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import matplotlib

In [ ]:
# Fill in this cell with your personal details:
# - Name: xxx
# - Student ID: xxx
# - Email: xxx

Deadline: December 18, 2026.

Instructions:
- Submit your completed notebook on Gradescope.
- All cells must be executable and their outputs should not be erased before submission.
- AI assistants are allowed, without restriction. Everything you submit, you must be able to explain: if you cannot explain or justify a line of your code or a choice you made, you will lose points.
- This case study must be carried out alone. You are not allowed to discuss or collaborate with other students.
- During the January exam session, you will explain and defend your solution in a 30-minute oral discussion, separate from the oral exam on theory. You may be asked about any part of your code and choices, and to modify or extend your analysis.
- The case study, evaluated through this discussion, accounts for 40% of the final grade. The oral exam on theory accounts for the other 60%.
- A minimum of 8/20 is required for each component. Below that, the lowest of the two grades becomes the final grade.

Comments:
- Solve the case study below by following the Bayesian workflow as best as possible.
- Follow the data visualization principles to make your plots effective and readable.
- Make the best use of the Python scientific ecosystem.
- Add comments to explain your reasoning, choices, and interpretations.
- You can add additional cells as needed.
- Most questions are split into points A, B, C, and so on. Start your answer to each point with a markdown cell that holds only its title, for example `### QUESTION 2.A`. For a question without points, the title is `### QUESTION 1`.
- Write your code so that you can re-run the expensive cells without re-running everything, and use `jax` with `jax.lax.scan` for the recursions of Q8.

# Introduction

This case study is about one year of sea level in the harbour of Boston, on the east coast of the United States, measured a few times per day at irregular times. The sea there rises and falls by about three metres twice a day, and a few times a year a storm pushes it high enough to flood the streets near the harbour. The measured level is the sum of several effects that cannot be observed separately. Your job is to separate them, with a model that also says how sure it is about each one.

## What moves the sea

**1. The tide.** The Moon pulls on the Earth and its oceans, more strongly on the side facing it than on the far side. The ocean bulges in two places, one facing the Moon and one on the opposite side. As the Earth turns, a harbour passes through both bulges, and gets two high waters. The Moon also moves along its orbit, so the same harbour faces it again only every 24 h 50 min, and its high waters come every 12 h 25 min. This main part of the tide is called **M2**. Your measurements are too far apart to see it directly, but the periodogram of Question 2 will find it.

The Sun does the same, with about half the strength of the Moon. It comes back over the same place every 24 h, so its tide, called **S2**, has a period of exactly 12 h. The two tides go in and out of step every 14.77 days. In step, they add up and the tide is large; out of step, they partly cancel and the tide is small. The animation below shows this.

<img src="tides.gif" width="800">

Two more effects matter. The distance between the Earth and the Moon changes over 27.55 days, so the tide is larger when the Moon is closest and smaller when it is farthest. The orbit of the Moon is also tilted: the Moon is usually north or south of the equator, by up to 28 degrees. The two bulges are then centred north and south of the equator, and the two high waters of a day are unequal. This inequality vanishes every two weeks, when the Moon crosses the equator.

<img src="tilt_explained.png" width="900">

Waves with two cycles per day always give two equal high waters. Unequal high waters therefore need a wave with one cycle per day as well, the dotted curve in the figure. Your periodogram will show lines near one cycle per day, although the sea rises twice a day.

Each of these effects is one more sine wave with its own period, called a **constituent**. M2 is the largest almost everywhere. In Boston, the second largest comes from the distance of the Moon. The periods of the constituents follow from astronomy and are known precisely before seeing any data; in Question 4 you compute about ten of them from four astronomical numbers. Their size and timing depend on the shape of the coast and the depth of the sea: the same Moon gives a tide of 0.3 m in the Mediterranean, 3 m in Boston and 16 m in the Bay of Fundy. Size and timing must be learned from the data.

The harbour itself adds constituents. In shallow water the tide is no longer a pure sine wave, but it still repeats every 12 h 25 min. A signal that repeats every $T$ hours is a sum of sine waves with periods $T$, $T/2$, $T/3$, and so on (a Fourier series). The first of these extra waves, at 6 h 13 min, is called **M4** and is in the table of Question 4.

**2. The storm surge.** The weather moves the sea too. When the air pressure drops by 1 hPa, the sea rises by about 1 cm, so a storm at 980 hPa lifts the sea by about 33 cm compared with a normal 1013 hPa. Wind blowing towards the coast also pushes water against it. This rise of the sea caused by the weather is called the **surge**. It builds up over a few hours when a storm arrives, and goes back to zero over a day or two once the storm has passed. It does not repeat, so no calendar can predict it, and it is what floods harbours.

**3. The yearly cycle.** Water expands when it warms, and winds and sea currents change with the seasons. In Boston, the average sea level goes up and down by about 10 cm over the year. This cycle repeats with a known period, one year, so it is handled like a tidal constituent.

**4. The fast movements.** Waves and the water sloshing back and forth inside the harbour move the sea within an hour. With measurements several hours apart, they look like random noise from one measurement to the next, to which the error of the instrument adds a little. The noise term of the model collects both.

## Why this is a data science problem

The gauge measures only the sum of these four effects. No instrument measures the tide or the surge separately, so there is no ground truth for this split. The tide prediction published by the agency running the gauge, which you also get, is the output of another model.

The tide, the surge and the yearly level are latent states that change over time, and the measurement is their sum plus noise. This is the setting of the state-space models of Lecture 5. What you want is the posterior distribution of these states given the data.

The split matters in practice. Tide tables can be printed years in advance, because the tide repeats; flood warnings are about the surge, which does not. The highest water levels come from a storm that arrives on a day of large tide.

The classical method, in use for more than a century, fits a sum of sine waves at the known periods by least squares and calls what is left the surge. It gives no uncertainty on the surge, assumes that the size of each constituent never changes, and cannot forecast the surge. A state-space model does all three.

You will explore the data, build the model from the effects above, compute the posterior of the hidden components with the Kalman filter and smoother, estimate the parameters, separate the tide, the surge and the mean level, and finally check the model and forecast the surge.

# Data exploration

You are given the file `water-level-irregular.csv`. It contains one year of measurements from the tide gauge of Boston, run by the American agency NOAA as station 8443970, from 1 January to 31 December 2024. The measurements are not equally spaced. They were picked at random from the hourly record of the gauge, with 3 to 9 hours between two measurements, so every time is a whole hour. Times are given in UTC, so there is no change of hour for summer time.

**`x`** is the measured sea level, in metres. Each value is one measurement, taken at that time, and it is given as a height above a fixed reference level called the zero level of the station. NOAA computed this level once, as the average of all measurements made at this station between 1983 and 2001. Averaging over nineteen years removes the slow cycles of the Moon, the longest of which lasts 18.6 years. Sea level has risen since that period, so the average of `x` over 2024 is not zero. This shift is real, its value is not given to you, and your model will have to estimate it.

**`x_pred`** is the tide that NOAA predicts at the time of each measurement, in metres, computed from many years of past measurements at this station. **You are not allowed to use it to fit your model.** It is kept aside so that, at the end, you can compare the tide found by your model with the tide found by a completely different method. You may use it in Question 2 to remove the tide and look at what is left, as long as you say so.

**`p`** is the air pressure, in hPa, measured by a sensor on the gauge itself, so at the same place as the sea level. It is missing for a few measurements.

**`u` and `v`** are the wind speed towards the east and towards the north, in m/s. They come from a buoy about thirty kilometres off the coast, because the Boston gauge has no wind sensor. Your model will not use them. They are there to help you check the storms in Question 2.

<div class="alert alert-success">
    
**Q1**. Load the data from `water-level-irregular.csv` into a Pandas DataFrame.
    
</div>

<div class="alert alert-success">
    
**Q2**. Explore the dataset and report relevant insights. At least:

**A.** Plot the water level over four timescales: the whole year, two months, one week, and two days, with one dot per measurement. Comment on what each timescale reveals, and on what it no longer reveals now that the measurements are several hours apart.

**B.** Plot the periodogram of the record. The measurements are not equally spaced, so use a Lomb-Scargle periodogram, which fits a sine wave by least squares at each frequency. Identify the dominant lines and relate them to the lunar day (24.84 h) and the solar day (24 h). State the frequency resolution of your periodogram and what it implies about which lines you can and cannot separate.

**C.** Your periodogram shows lines near one cycle per day, although the sea rises twice a day. Explain where they come from.

**D.** Plot `x - x_pred` over the whole year. The predicted tide is removed, so what is left is mostly the weather. When is it largest? Use the pressure and the wind to check that the largest peaks really are storms. Find the measurement with the largest value, and the measurement with the highest water level `x`. Are they the same?

**E.** Check every column for missing values, and say where they are. Look also at the time between two consecutive measurements.
    
</div>

# Build: a state-space model of the water level

## Continuous time

We first describe the system in continuous time, and discretize it afterwards, as recommended in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#continuous-or-discrete).

The **tide** is written as in the introduction, as a sum of sine waves, one per constituent. With $J$ constituents,
$$\text{tide}(t) = \sum_{j=1}^{J} \alpha_j \cos(\omega_j t - \phi_j),$$
where constituent $j$ has a known angular frequency $\omega_j$, which you will compute in Question 4, an unknown amplitude $\alpha_j$, its size, and an unknown phase $\phi_j$, its timing. Which constituents to include, and therefore the value of $J$, is your choice in Question 5.

To use the tide in a state-space model, each constituent must be written as in Lecture 5, as a state that moves from one step to the next by a linear rule, plus a little noise. A cosine is not directly of that form, but it becomes so with a simple picture.

Imagine a point that turns around a circle of radius $\alpha_j$ at constant speed, one full turn per period of the constituent. At time $t$ its angle is $\omega_j t - \phi_j$. Its horizontal coordinate is
$$c_j(t) = \alpha_j \cos(\omega_j t - \phi_j),$$
which is exactly the contribution of constituent $j$ to the water level. Its vertical coordinate is
$$e_j(t) = \alpha_j \sin(\omega_j t - \phi_j).$$

<img src="rotating_point.png" width="900">

The water level only uses $c_j$. We still need $e_j$, because $c_j$ alone does not say where the point goes next. The same horizontal position is reached once while $c_j$ goes up and once while it goes down, and only $e_j$ tells which.

The pair $r_j(t) = (c_j(t), e_j(t))$ is the position of the point, and it contains the size and the timing of the constituent. The size $\alpha_j$ is the distance of the point to the centre, and the timing $\phi_j$ fixes where the point is on the circle at a given time. So $\alpha_j$ and $\phi_j$ do not need to be parameters of the model. We put $r_j$ in the state, the Kalman filter estimates the position of the point, and the size and the timing come with it.

Time is measured in hours, so $\omega_j = 2\pi / (\text{period in hours})$, in radians per hour. For M2, $\omega_j = 2\pi / 12.42$. Taking the derivative of $c_j$ and $e_j$ with respect to time gives $\frac{dc_j}{dt} = -\omega_j e_j$ and $\frac{de_j}{dt} = \omega_j c_j$, a linear rule. Adding small random pushes, the equation of the turning point in continuous time is
$$dr_j(t) = \Omega_j \, r_j(t) \, dt + \sigma_\text{tide} \, dW(t), \qquad \Omega_j = \begin{bmatrix} 0 & -\omega_j \\ \omega_j & 0 \end{bmatrix}.$$
The first term turns the point at speed $\omega_j$. The second term gives it small random pushes, which slowly change its distance to the centre and its position on the circle, so the size and the timing of the constituent can drift a little during the year.

With $\sigma_\text{tide} = 0$ there are no pushes, and the point turns forever on the same circle. This is a sine wave of fixed size and timing, as in the classical method of the introduction.

The **surge** $s(t)$, the rise of the sea caused by the weather, is pushed away from zero when a storm arrives and relaxes back once it has passed. This is the same Ornstein-Uhlenbeck process that described the wolf in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#ou-process), in one dimension and with equilibrium point $0$,
$$ds(t) = -\kappa \, s(t) \, dt + \sigma_s \, dW(t),$$
where $1/\kappa$ [h] is the relaxation time of the surge, the time it needs to shrink by a factor $e$ once the weather is calm, and $\sigma_s^2 / (2\kappa)$ is its variance in the long run.

In the model, the effect of the air pressure is written separately, as $\beta \tilde{p}_t$ in the observation below. So $s(t)$ is the part of the surge that the local air pressure does not explain, for example the push of the wind, and the total surge is $s(t) + \beta \tilde{p}(t)$.

The **yearly cycle** is the slow rise and fall of the average sea level over the year, described in the introduction. It repeats every year, so it can be written exactly like the tide, as sine waves with known periods. One sine wave with a period of one year gives a very regular shape, six months up and six months down. The real yearly cycle does not have to be that regular, so we add a second sine wave with a period of half a year. Together, the two can make a shape that rises faster than it falls, or stays high for longer than it stays low. This is the same idea as the shallow water effect of the introduction, with periods $T$ and $T/2$. These two sine waves are simply two more constituents in the sum above, each with its own turning point. They follow the same equation as the other constituents, $dr_j(t) = \Omega_j \, r_j(t) \, dt + \sigma_\text{tide} \, dW(t)$, with $\omega_j = 2\pi / (1 \text{ year})$ for the first and $\omega_j = 2\pi / (\text{half a year})$ for the second. Only the speed of the turning point changes. In Question 4 they are called Sa and Ssa.

The **mean level** is a constant $\ell$, the shift between the zero level of the station and the average level of the sea in 2024. Since it does not change, its equation is simply $d\ell(t) = 0$. A mean level that drifts slowly over time would be a natural alternative, but it is not used here.

Each $dW(t)$ above is its own Brownian motion, independent of all the others, and the two coordinates of each $r_j$ receive independent pushes.

## Discrete time

The data arrive at irregular times. In the continuous equations above, $t$ was the time in hours. From now on, $t$ counts the measurements, and $\Delta t$ is the time between measurement $t-1$ and measurement $t$, between 3 and 9 h. We now write how each component moves from one measurement to the next. $\Delta t$ changes from one step to the next, so the transitions below change too, as for the wolf in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#ou-discretization). Each equation above can be solved exactly over one step, with no approximation. For the surge, this is the exact discretization of the Ornstein-Uhlenbeck process in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#ou-discretization), with $\mu = 0$. For the tide, the solution is a rotation. You do not need to derive it, it is given below.

- **Tide.** In one step, each turning point turns by the angle $\omega_j \Delta t$, and the random pushes add a small noise,
$$r_{j,t} = \text{Rot}(\omega_j \Delta t)\, r_{j,t-1} + w_{j,t}, \qquad w_{j,t} \sim \mathcal{N}\left(0, \sigma_\text{tide}^2 \Delta t \, I_2\right), \qquad \text{Rot}(a) = \begin{bmatrix} \cos a & -\sin a \\ \sin a & \cos a \end{bmatrix}.$$
This holds for all $J$ constituents, including the two of the yearly cycle.
- **Surge.** In one step, the surge keeps only a fraction $e^{-\kappa \Delta t}$ of its value and receives a new random push,
$$s_t = e^{-\kappa \Delta t}\, s_{t-1} + \eta_t, \qquad \eta_t \sim \mathcal{N}\left(0, \frac{\sigma_s^2}{2\kappa}\left(1 - e^{-2\kappa \Delta t}\right)\right).$$
- **Mean level.** It does not move, $\ell_t = \ell_{t-1}$.

Stacking all the components into one vector, the latent state
$$z_t = (c_{1,t}, e_{1,t}, \ldots, c_{J,t}, e_{J,t}, s_t, \ell_t) \in \mathbb{R}^{2J + 2},$$
the three transitions above (tide, surge, mean level) become a single linear Gaussian transition, as in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#lgssm) with $b = 0$,
$$p(z_t | z_{t-1}) = \mathcal{N}(z_t | A z_{t-1}, Q),$$
where $A$ and $Q$ are computed with the $\Delta t$ of that step, $A$ contains the rotations, the factor $e^{-\kappa \Delta t}$ and a $1$ for the mean level, and $Q$ contains the noise variances.

The measured water level $x_t$ is the sum of the components, plus the effect of the air pressure and a noise term that collects the fast movements and the error of the instrument,
$$x_t \mid z_t \sim \mathcal{N}(H z_t + \beta \tilde{p}_t, R), \qquad R = \sigma_x^2.$$
Here $H$ is a single row, since $x_t$ is a single number, with one entry per component of $z_t$. It has a $1$ in front of each $c_{j,t}$, of $s_t$ and of $\ell_t$, and a $0$ in front of each $e_{j,t}$, which is not part of the water level, so that
$$H z_t = \sum_{j=1}^{J} c_{j,t} + s_t + \ell_t.$$
The air pressure minus its average over the year is $\tilde{p}_t = p_t - \bar{p}$ [hPa], a known input called a covariate, and $\beta$ [m/hPa] is an unknown parameter.

The state contains exactly what is hidden and needed, either to build the water level now or to move it forward to the next measurement. Each $c_{j,t}$ is part of the water level. Each $e_{j,t}$ is not, but without it $c_{j,t}$ cannot be moved forward, since we would not know whether it goes up or down. The surge $s_t$ is hidden and has memory, since a surge at one moment is still partly there a few hours later. The mean level $\ell_t$ is unknown, and putting it in the state lets the Kalman filter estimate it. The air pressure is not in the state, because it is measured, and neither is the observation noise, because it has no memory from one measurement to the next.

The parameters of the model are
$$\theta = (\kappa, \sigma_s, \sigma_\text{tide}, \sigma_x, \beta).$$

<div class="alert alert-success">
    
**Q3**. Build the model in code.

**A.** Start with a single constituent, $J = 1$, so that $z_t = (c_t, e_t, s_t, \ell_t)$. Write $A$ and $Q$ by hand, as $4 \times 4$ matrices. Each row of $A$ says how one component of the new state $z_t$ is made from the components of the old state $z_{t-1}$, and $Q$ holds the variances of the noises.

**B.** Write a function `build_A_Q(theta, dt, omega)` that returns $A$ and $Q$ for any $J$ and a time step `dt`, using the exact transitions given above, as [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#ou-discretization) does for the wolf, and a function `build_H(J)` that returns $H$, following the transitions and the order of $z_t$ given above. Here `omega` holds the angular frequencies $\omega_j = 2\pi / (\text{period in hours})$ of the constituents. You will compute them precisely in Question 4. For now, test your functions with M2 and S2 only, using the periods given in the introduction, 12 h 25 min and 12 h, and with any values of $\theta$ you like, for example $\kappa = 0.05$, $\sigma_s = 0.03$, $\sigma_\text{tide} = 0.002$, $\sigma_x = 0.03$ and $\beta = -0.01$. Print $A$, $Q$ and $H$, and explain why $A$ is block diagonal.

**C.** In [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#ou-process), the wolf is discretized with Euler-Maruyama, new state = old state + rate of change times $\Delta t$, that is $z_t = z_{t-1} + f(z_{t-1})\, \Delta t + w_t$. For the turning point of M2, the rate of change is $\Omega_j r$, so without noise the Euler-Maruyama step is $r_t = r_{t-1} + \Omega_j r_{t-1} \Delta t = (I + \Omega_j \Delta t)\, r_{t-1}$. The Euler-Maruyama matrix $I + \Omega_j \Delta t$ therefore takes the place of the exact rotation $\text{Rot}(\omega_j \Delta t)$. Build $A$ with your function for M2 alone, then a second matrix where the M2 block is replaced by $I + \Omega_j \Delta t$. Starting from $c = 1.3$ m and $e = s = \ell = 0$, apply each matrix hour after hour for three days, without noise, and plot $c_t$ for both. What happens? Euler-Maruyama works well for the wolf, so why not here? Think about how much each state moves during one step, and about the shape of its motion, a decay towards home for the wolf and a turn around a circle for the tide.

**D.** Check in code that two steps of half an hour give exactly one step of one hour, that is $A(\Delta t/2)^2 = A(\Delta t)$ and $A(\Delta t/2)\, Q(\Delta t/2)\, A(\Delta t/2)^T + Q(\Delta t/2) = Q(\Delta t)$. The second formula is the prediction step of the Kalman filter of [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#kalman-filter-proof), $P^-_t = A P_{t-1} A^T + Q$, applied to the noise of the first half step. What does this mean for measurements that are 3 to 9 hours apart? Does the Euler-Maruyama step of M2 pass the same check?
    
</div>

<div class="alert alert-success">
    
**Q4**. Compute the frequencies of the constituents.

The frequencies of the constituents are not estimated from the data. They come from the motions of the Earth, the Moon and the Sun, which are known very precisely, and each constituent combines two of these motions. Work with frequencies $f$ in cycles per day, $f = 1/\text{period}$.

| motion | period |
| --- | --- |
| the Earth turns once with respect to the Sun | 24 h |
| the Earth goes once around the Sun | 365.2422 days |
| the Moon goes once around the Earth | 27.321582 days |
| the distance between the Earth and the Moon goes through one cycle | 27.554550 days |

| constituent | frequency |
| --- | --- |
| S2 | $2 f_\text{day}$ |
| K1 | $f_\text{day} + f_\text{year}$ |
| P1 | $f_\text{day} - f_\text{year}$ |
| K2 | $2 f_\text{K1}$ |
| M2 | $2 (f_\text{K1} - f_\text{Moon})$ |
| N2 | $f_\text{M2} - f_\text{distance}$ |
| O1 | $f_\text{M2} - f_\text{K1}$ |
| M4 | $2 f_\text{M2}$ |
| Sa | $f_\text{year}$ |
| Ssa | $2 f_\text{year}$ |

K1 makes one cycle per turn of the Earth with respect to the stars, which takes 23 h 56 min. This is a little shorter than a day, because the Earth also moves around the Sun. M2 makes two cycles per lunar day, the 24 h 50 min of the introduction.

These frequencies give the $\omega_j$ of the Build, the `omega` you pass to `build_A_Q`, with $\omega_j = 2\pi f_j / 24$ in rad/h.

**A.** Compute, for the ten constituents, the frequency in cycles per day, the period in hours and $\omega_j$ in rad/h, and show them in a table. Check that M2 and S2 give the periods of the introduction.

**B.** Draw the ten frequencies as vertical lines on your periodogram of Question 2. Does each one fall on a peak?

**C.** Why not simply read the frequencies from the periodogram? Suppose the frequency of M2 you read is wrong by half the frequency resolution of your periodogram, $1/366$ cycle per day. After one year, by how many hours would your M2 be early or late? Compare with the period of M2.
    
</div>


<div class="alert alert-success">
    
**Q5**. Choose the constituents and simulate from the model.

**A.** Two constituents can only be separated by a record longer than $1/|f_1 - f_2|$. Compute that length for the pairs M2/S2, M2/N2, S2/K2, K1/P1 and Sa/Ssa, and decide which constituents you keep. Justify your choice against the record you actually have, and say what happens to a constituent you drop, knowing that it does not disappear from the data.

**B.** Sa deserves a separate comment. The record is exactly one year long, so it contains exactly one cycle of Sa. Say what that implies for the separation between the mean level $\ell$ and Sa, and what you would need in order to separate them.

**C.** Write functions to sample from $p(z_t|z_{t-1})$ for a given $\Delta t$ and from $p(x_t|z_t)$, and a function to simulate a full trajectory $z_{1:T}, x_{1:T}$ at the measurement times of the data, given $\theta$ and an initial distribution $p(z_1)$. The observation needs the air pressure at every step, but the pressure has a few missing values, found in Q2. Decide how to fill them, and say why.
    
</div>


<div class="alert alert-success">
    
**Q6**. Define a prior $p(z_1) = \mathcal{N}(z_1 | m_1^-, P_1^-)$ over the initial state, and a prior over $\sigma_x$. The filter of Q8 starts from $p(z_1)$, before it has seen any data.

**A.** $P_1^-$ has to be wide enough to contain any plausible size and timing of the constituents, and, on $\ell$, the shift between the zero level of the station and the mean of the record, which you can read off the data. Choose $m_1^-$ and $P_1^-$, and justify each value.

**B.** Why is it safer to choose $P_1^-$ too wide than too narrow?

**C.** With measurements several hours apart, the data alone say little about $\sigma_x$, so its prior will be used in Q9 to estimate it. $\sigma_x$ is not the accuracy of the gauge, which is far better than anything you will estimate. Say what it actually absorbs in a record of measurements several hours apart, and choose its prior accordingly.
    
</div>


<div class="alert alert-success">
    
**Q7**. Draw the probabilistic graphical model. What are the observed variables? What are the latent variables? What are the parameters? Which quantities are fixed and known? Write the factorization of the joint distribution and count how many latent variables the model has.
    
</div>

# Compute

The model is a linear Gaussian state-space model, so the Bayes filter and the Bayes smoother of Lecture 5 are available in closed form, as the [Kalman filter](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#kalman-filter) and the [Kalman smoother](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#kalman-smoother).

<div class="alert alert-success">
    
**Q8**. Implement the Kalman filter and the Rauch-Tung-Striebel smoother.

**A.** Implement the Kalman filter. It returns the filtering distributions $p(z_t|x_{1:t}) = \mathcal{N}(z_t|m_t, P_t)$, the one-step-ahead predictions $\mathcal{N}(z_t|m_t^-, P_t^-)$, and the innovations $\nu_t = x_t - Hm_t^- - \beta\tilde{p}_t$ with their variances $S_t = HP_t^-H^T + R$. Use `jax` and `jax.lax.scan`. The state has dimension $2J + 2$, the record has 1443 steps, and you will run the filter many times in Q9, when you estimate the parameters.

**B.** Implement the smoother. It returns the smoothing distributions $p(z_t|x_{1:T}) = \mathcal{N}(z_t|m_t^s, P_t^s)$.

**C.** The measurements are not equally spaced. Use the time between two consecutive measurements as $\Delta t$, so that $A$ and $Q$ change from one step to the next. Also let the filter skip the update at a step without observation, and say why this is the right thing to do. This lets you add steps at hours without data and get the water level there. Do not impute values.

**D.** A missing pressure is not the same thing as a missing water level. Say why, and whether the way you filled the pressure in Q5 is right for the filter.

**E.** Validate your implementation before touching the real data. Simulate $x_{1:T}$ from your own model with a known $\theta$, run the filter and the smoother, and check that the latent states are recovered within the reported uncertainty. Report the empirical coverage of the 95% credible intervals of the surge.
    
</div>

<div class="alert alert-success">
    
**Q9**. Estimate the parameters and separate the components.

**A.** Write down the log likelihood $\log p(x_{1:T}|\theta)$. Using the decomposition $p(x_{1:T}|\theta) = \prod_{t=1}^{T} p(x_t | x_{1:t-1}, \theta)$ of [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#learning-parameters), and the fact that each factor is Gaussian in the linear Gaussian case, show that $$\log p(x_{1:T}|\theta) = -\frac{1}{2}\sum_{t}\left(\log 2\pi S_t + \frac{\nu_t^2}{S_t}\right),$$ the sum running over the observed time steps. Explain in one sentence what has happened to the latent states $z_{1:T}$.

**B.** Estimate $\theta$ by maximizing this log likelihood plus the log prior of $\sigma_x$ from Q6, with the gradient given by `jax`. Fit the positive parameters on a log scale.

**C.** Plot the log likelihood as a function of $\kappa$, with the other parameters fixed at their estimates, as in [Lecture 5](https://glouppe.github.io/dats0001-foundations-of-data-science/?p=lecture5.md#likelihood-kappa). What does it tell you about how well $\kappa$ is known?

**D.** Run the smoother with the estimated $\theta$, and plot on a common time axis the measured water level, the smoothed tide, the sum of $\mathbb{E}[c_{j,t}|x_{1:T}]$ over the tidal constituents, the smoothed surge $\mathbb{E}[s_t|x_{1:T}] + \beta \tilde{p}_t$ with its 95% band, and the smoothed slow level, the mean level $\ell$ plus the yearly cycle of Sa and Ssa, with its 95% band. Use the steps without observation of Q8 to draw the components at every hour, not only at the measurement times. Show the whole year, and the two days around the highest water level found in Q2.

**E.** Compare your tide plus yearly cycle with `x_pred`, which contains both. Report the RMS difference and say when the differences are largest. Compare $\beta$ with the rise of about 1 cm per hPa of pressure drop given in the introduction.
    
</div>


# Critique

<div class="alert alert-success">

**Q10**. Criticize the model.

**A.** Under the model, the standardized innovations $\nu_t / \sqrt{S_t}$ are independent draws from $\mathcal{N}(0, 1)$. Check this on the data with a histogram, a QQ-plot and their autocorrelation. When does the model fail, and why?

**B.** Simulate a few records from the fitted model at the measurement times, and show them next to the data on the plots of Q2. What does the model reproduce, and what does it miss?

</div>

<div class="alert alert-success">

**Q11**. Forecast the surge.

Run the filter on the data up to two days before the highest water level found in Q2. From there, predict the surge $s_t + \beta \tilde{p}_t$ hour by hour over the next 48 hours with the prediction step alone, with its 95% band, taking the measured pressure as if it came from a weather forecast. Compare with the surge estimated by the smoother from the whole record. Why does the band grow with the horizon?

</div>